# Dataset Import Experiment

Author: Luca Cotti (<luca.cotti@unibs.it>)

This notebook contains an experiment for importing the "russellmitchell" dataset from the AIT-LDS repository.


## Setup

In [1]:
import json
import os
import re
import shutil
from pathlib import Path

import polars as pl
from thefuzz import fuzz

# Delete imported data if it already exists
DELETE_OUT_IF_EXISTS = True

# Path of the dataset to import
DATASET_PATH = "../../russellmitchell"

# These log dir contain just the logs on user/attacker behavior generation
LOG_DIRS_TO_IGNORE = ["attacker", "ext_user", "internal_employee", "remote_employee"]

# Maximum number of lines to read from each log file
TRUNC_LINES = 10000

# Levenshtein similarity threshold for log comparisong
SIMILARITY_THRESHOLD = 90

# Where to store the processed csv files
OUT_DIR = "./out_full"

In [2]:
from typing import NamedTuple


class TTP(NamedTuple):
    """A MITRE ATT&CK TTP representation."""

    tactic: str
    techniques: list[str]


# Map the log files labels to TTPs, using the description provided in the paper
LABELS_TO_TTPS: dict[str, TTP] = {
    "['attacker_vpn', 'foothold']": TTP("Initial Access", ["Valid Accounts"]),
    "['service_scan', 'foothold']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['attacker_http', 'foothold', 'service_scan']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['traceroute', 'foothold']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['attacker_http', 'foothold', 'dirb']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Host Information"],
    ),
    "['attacker_http', 'foothold', 'wpscan']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Host Information"],
    ),
    "['attacker_http', 'foothold', 'webshell_upload']": TTP(
        "Execution",
        [
            "Exploitation for Client Execution Persistence",
            "Server Software Component Discovery",
        ],
    ),
    "['attacker_http', 'foothold', 'webshell_cmd']": TTP(
        "Persistence",
        ["Server Software Component Discovery"],
    ),
    "['webshell_cmd', 'escalate']": TTP("Credential Access", ["OS Credential Dumping"]),
    "['escalate', 'crack_passwords']": TTP(
        "Credential Access",
        ["Brute Force: Password Cracking"],
    ),
    "['attacker_change_user', 'escalate']": TTP("Privilege Escalation", ["Valid Accounts"]),
    "['attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command']": TTP(
        "Privilege Escalation",
        ["Valid Accounts"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalate']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalate', 'escalated_sudo_session']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalated_sudo_session', 'escalate']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['dnsteal', 'exfiltration-service', 'attacker']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
    "['dnsteal', 'attacker', 'dnsteal-received']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
    "['dnsteal', 'attacker', 'dnsteal-dropped']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
}

## Load dataset

In [3]:
from itertools import islice

# List all directories in the logs subdir of the DATASET_PATH
logs_subdir_path = Path(DATASET_PATH) / "gather"
logs_dirs = [d for d in Path.iterdir(logs_subdir_path) if Path.is_dir(d)]

labels_subdir_path = Path(DATASET_PATH) / "labels"

# Filter out directories containing "attacker", "ext_user", or "Internal_employee"
devices = [d.name for d in logs_dirs if not any(re.search(x, d.name) for x in LOG_DIRS_TO_IGNORE)]


def read_log_and_labels(device: str, app: str, file_name: str) -> pl.DataFrame:
    """Read a log file and returns a dataframe with the lines and line numbers."""
    file_path = Path(logs_subdir_path) / device / "logs" / app / file_name

    with file_path.open("r") as f:
        lines = [line.strip() for line in islice(f, TRUNC_LINES)]

    log_df = pl.DataFrame({"text": lines}).with_columns(
        [
            pl.arange(0, pl.len()).alias("line_number"),
            pl.lit(device).alias("device"),
            pl.lit(file_name).alias("file_name"),
            pl.lit("").alias("tactic"),
            pl.lit("").alias("techniques"),
        ],
    )

    labels_path = labels_subdir_path / device / "logs" / app / file_name

    if Path.exists(labels_path):
        with labels_path.open("r") as file:
            rows = [json.loads(line.strip()) for line in file]

        if rows:
            max_line = log_df.height
            rows = [r for r in rows if r.get("line", -1) < max_line]

            if rows:
                line_nums = [r["line"] for r in rows]
                tactics = []
                techniques = []
                for r in rows:
                    tactic, techs = LABELS_TO_TTPS[str(r["labels"])]
                    tactics.append(tactic)
                    techniques.append("[" + ", ".join(techs) + "]")

                labels_df = pl.DataFrame(
                    {
                        "line_number": line_nums,
                        "tactic_label": tactics,
                        "techniques_label": techniques,
                    },
                )

                # left join labels onto log_df and coalesce to keep original values when no label
                log_df = (
                    log_df.join(labels_df, on="line_number", how="left")
                    .with_columns(
                        [
                            pl.coalesce(pl.col("tactic_label"), pl.col("tactic")).alias("tactic"),
                            pl.coalesce(pl.col("techniques_label"), pl.col("techniques")).alias("techniques"),
                        ],
                    )
                    .drop(["tactic_label", "techniques_label"])
                )

    return log_df


def import_data() -> None:
    """Import the dataset and process the logs."""
    print("\nProcessing logs...")

    for device in devices:
        print(f"  - {device}")

        # Go to the logs subdir of the current device
        logs_dir = Path(logs_subdir_path) / device / "logs"

        # Walk through the logs directory and read all log files
        for root, _, files in os.walk(logs_dir):
            # The log app corresponds to the relative path from the logs directory.
            # If the log is a system log, the log app will be empty.
            app = os.path.relpath(root, logs_dir) if root != logs_dir else ""

            for file_name in files:
                if "log" in file_name:
                    print(f"    > {file_name}...", end="")

                    # Binary files should be skipped.
                    # If the file is binary, an exception will be raised.
                    try:
                        log_df = read_log_and_labels(device, app, file_name)
                    except UnicodeDecodeError:
                        continue

                    if all(log_df["tactic"] == ""):
                        print("No tactics, skipping.")
                        continue

                    # keep only labeled rows
                    log_df = log_df.filter(pl.col("tactic") != "")

                    # deduplicate by fuzzy text similarity (keep first occurrence)
                    file_logs = []
                    seen_texts = []
                    for row in log_df.to_dicts():
                        print(row["text"])
                        if any(
                            fuzz.partial_token_sort_ratio(row["text"], t) > SIMILARITY_THRESHOLD for t in seen_texts
                        ):
                            continue
                        file_logs.append(row)
                        seen_texts.append(row["text"])

                    log_df = pl.DataFrame(file_logs)

                    Path.mkdir(Path(OUT_DIR), exist_ok=True, parents=True)

                    out_path = Path(OUT_DIR) / "out.csv"
                    append = out_path.exists() and out_path.stat().st_size > 0
                    with out_path.open("a", encoding="utf-8", newline="") as f:
                        log_df.write_csv(f, include_header=not append)

                    # Delete the dataframe to free up memory
                    del log_df

                    print("✔")

    print("Done.")

In [4]:
if Path(OUT_DIR).exists() and DELETE_OUT_IF_EXISTS:
    print("Deleting existing output directory...")
    shutil.rmtree(OUT_DIR)

if not Path(OUT_DIR).exists():
    import_data()
else:
    print("Output directory already exists. Skipping data import.")

Deleting existing output directory...

Processing logs...
  - intranet_server
    > syslog.2...No tactics, skipping.
    > syslog.4...No tactics, skipping.
    > auth.log...Jan 24 04:37:40 intranet-server su[27950]: + /dev/pts/1 www-data:jhall
Jan 24 04:37:40 intranet-server su[27950]: pam_unix(su:session): session opened for user jhall by (uid=33)
Jan 24 04:37:40 intranet-server systemd-logind[957]: New session c1 of user jhall.
Jan 24 04:37:58 intranet-server sudo:    jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellmitchell.com/wp-content/uploads/2022/01 ; USER=root ; COMMAND=list
Jan 24 04:38:06 intranet-server sudo:    jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellmitchell.com/wp-content/uploads/2022/01 ; USER=root ; COMMAND=/bin/cat /etc/shadow
Jan 24 04:38:06 intranet-server sudo: pam_unix(sudo:session): session opened for user root by (uid=0)
Jan 24 04:38:06 intranet-server sudo: pam_unix(sudo:session): session closed for user root
Jan 24 04:39:01 intranet-server